# Disease Prediction from Medical Data

**CodeAlpha Machine Learning Internship - Task 4**

This project predicts whether a breast tumor is **Malignant** or **Benign** using structured medical data and classification algorithms.


## 1. Import Required Libraries

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings("ignore")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("dataset.csv")
df.head()

## 3. Dataset Information

In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

## 4. Data Cleaning

In [ ]:
df = df.drop_duplicates()

for column in df.columns:
    if df[column].dtype == "object":
        df[column] = df[column].fillna(df[column].mode()[0])
    else:
        df[column] = df[column].fillna(df[column].median())

print("Data cleaning completed.")
print("New shape:", df.shape)

## 5. Diagnosis Distribution

In [ ]:
os.makedirs("images", exist_ok=True)

df["Diagnosis"].value_counts().plot(kind="bar")
plt.title("Diagnosis Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("images/dataset_distribution.png", dpi=200)
plt.show()

## 6. Correlation Heatmap

In [ ]:
X_numeric = df.drop("Diagnosis", axis=1)
corr = X_numeric.corr()

plt.figure(figsize=(12, 10))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=6)
plt.yticks(range(len(corr.columns)), corr.columns, fontsize=6)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("images/correlation_heatmap.png", dpi=200)
plt.show()

## 7. Prepare Features and Target

In [ ]:
X = df.drop("Diagnosis", axis=1)
y = df["Diagnosis"].map({"Malignant": 0, "Benign": 1})

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

## 8. Train-Test Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 9. Train Classification Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

results = []

for model_name, model in models.items():
    if model_name in ["Logistic Regression", "SVM"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_probability = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_probability = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_probability)
    })

results_df = pd.DataFrame(results)
results_df

## 10. Accuracy Comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["Accuracy"])
plt.title("Model Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("images/accuracy_comparison.png", dpi=200)
plt.show()

## 11. Select Best Model

In [ ]:
best_model_name = results_df.sort_values(
    "Accuracy", ascending=False
).iloc[0]["Model"]

best_model = models[best_model_name]

if best_model_name in ["Logistic Regression", "SVM"]:
    y_pred_best = best_model.predict(X_test_scaled)
    y_probability_best = best_model.predict_proba(X_test_scaled)[:, 1]
else:
    y_pred_best = best_model.predict(X_test)
    y_probability_best = best_model.predict_proba(X_test)[:, 1]

print("Best Model:", best_model_name)

## 12. Classification Report

In [ ]:
print(classification_report(
    y_test,
    y_pred_best,
    target_names=["Malignant", "Benign"]
))

## 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.colorbar()
plt.xticks([0, 1], ["Malignant", "Benign"], rotation=20)
plt.yticks([0, 1], ["Malignant", "Benign"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("images/confusion_matrix.png", dpi=200)
plt.show()

## 14. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_probability_best)
auc_score = roc_auc_score(y_test, y_probability_best)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"{best_model_name} (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig("images/roc_curve.png", dpi=200)
plt.show()

print("ROC-AUC Score:", auc_score)

## 15. Feature Importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
elif hasattr(best_model, "coef_"):
    importances = np.abs(best_model.coef_[0])
else:
    importances = models["Random Forest"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values("Importance", ascending=False).head(15)

plt.figure(figsize=(10, 7))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("images/feature_importance.png", dpi=200)
plt.show()

importance_df

## 16. Save Best Model

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(best_model, "models/disease_prediction_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(list(X.columns), "models/feature_names.pkl")

print("Model saved successfully.")
print("models/disease_prediction_model.pkl")
print("models/scaler.pkl")
print("models/feature_names.pkl")

## 17. Test One Prediction

In [ ]:
sample = X_test.iloc[0:1]

if best_model_name in ["Logistic Regression", "SVM"]:
    sample_input = scaler.transform(sample)
else:
    sample_input = sample

prediction = best_model.predict(sample_input)[0]
result = "Benign" if prediction == 1 else "Malignant"

print("Predicted Diagnosis:", result)
print("Actual Diagnosis:", "Benign" if y_test.iloc[0] == 1 else "Malignant")

## 18. Conclusion

This project used structured medical data to predict whether a breast tumor is malignant or benign.  
Logistic Regression, SVM, Decision Tree, and Random Forest models were trained and evaluated.  
The best model was selected using accuracy and saved for future prediction.
